# Imports and setup

In [40]:
import pandas as pd
import numpy as np
from scipy.stats import randint, uniform
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, recall_score, precision_score, make_scorer
from sklearn.model_selection import train_test_split, cross_validate, StratifiedKFold, RandomizedSearchCV
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier

from churn.config import SEED, TARGET, RAW_PATH
from churn.data import load_raw, clean
from churn.features import build_preprocessor
from churn.models import train_mlp, predict_proba

# Load and clean

In [41]:
df = clean(load_raw(RAW_PATH))

# Train/test split

In [42]:
X = df.drop(columns=[TARGET])
y = df[TARGET]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=SEED
)

# Dummy baseline

In [43]:
make_scorer(precision_score, zero_division=0)
scoring = ["roc_auc", "recall", "precision"]

dummy = DummyClassifier(strategy="prior")

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
res = cross_validate(dummy, X_train, y_train, cv=cv, scoring=scoring)
print({k: round(v.mean(), 3) for k, v in res.items() if k.startswith("test_")})

{'test_roc_auc': np.float64(0.5), 'test_recall': np.float64(0.0), 'test_precision': np.float64(0.0)}


d:\Practice\Data-Scientist\Churn-Prediction\.venv\Lib\site-packages\sklearn\metrics\_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
d:\Practice\Data-Scientist\Churn-Prediction\.venv\Lib\site-packages\sklearn\metrics\_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
d:\Practice\Data-Scientist\Churn-Prediction\.venv\Lib\site-packages\sklearn\metrics\_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()}

# Logistic Regression baseline

In [44]:
model = Pipeline([
    ("prep", build_preprocessor()),
    ("clf", LogisticRegression(max_iter=1000, random_state=SEED))
])
res = cross_validate(model, X_train, y_train, cv=cv, scoring=scoring)
print({k: round(v.mean(), 3) for k, v in res.items() if k.startswith("test_")})

{'test_roc_auc': np.float64(0.858), 'test_recall': np.float64(0.575), 'test_precision': np.float64(0.665)}


# Random Forest

## Basic

In [45]:
rf = Pipeline([
    ("prep", build_preprocessor()),
    ("clf", RandomForestClassifier(n_estimators=300, random_state=SEED, n_jobs=-1))
])
res = cross_validate(rf, X_train, y_train, cv=cv, scoring=scoring)
print({k: round(v.mean(), 3) for k, v in res.items() if k.startswith("test_")})

{'test_roc_auc': np.float64(0.83), 'test_recall': np.float64(0.497), 'test_precision': np.float64(0.63)}


## Hyperparameter tuning

In [46]:
rf_params = {
    "clf__n_estimators": randint(200, 600),
    "clf__max_depth": [None, 6, 10, 15],
    "clf__min_samples_leaf": randint(1, 20),
    "clf__max_features": ["sqrt", 0.3, 0.5],
}

rf_search = RandomizedSearchCV(
    rf, rf_params, n_iter=30, cv=cv, scoring="roc_auc", random_state=SEED, n_jobs=1
)
rf_search.fit(X_train, y_train)
print(rf_search.best_params_, round(rf_search.best_score_, 3))

{'clf__max_depth': 10, 'clf__max_features': 0.5, 'clf__min_samples_leaf': 14, 'clf__n_estimators': 280} 0.861


# XGBoost

## Basic

In [47]:
xgb = Pipeline([
    ("prep", build_preprocessor()),
    ("clf", XGBClassifier(n_estimators=300, learning_rate=0.05, max_depth=4,
                          random_state=SEED, n_jobs=-1)),
])
res = cross_validate(xgb, X_train, y_train, cv=cv, scoring=scoring)
print({k: round(v.mean(), 3) for k, v in res.items() if k.startswith("test_")})

{'test_roc_auc': np.float64(0.858), 'test_recall': np.float64(0.553), 'test_precision': np.float64(0.663)}


## Hyperparameter tuning

In [48]:
xgb_params = {
    "clf__n_estimators": randint(100, 600),
    "clf__learning_rate": uniform(0.01, 0.19),
    "clf__max_depth": randint(2, 7),
    "clf__min_child_weight": randint(1, 10),
    "clf__subsample": uniform(0.6, 0.4),
    "clf__colsample_bytree": uniform(0.6, 0.4),
}

xgb_search = RandomizedSearchCV(
    xgb, xgb_params, n_iter=30, cv=cv, scoring="roc_auc", random_state=SEED, n_jobs=1
)
xgb_search.fit(X_train, y_train)
print(xgb_search.best_params_, round(xgb_search.best_score_, 3))

{'clf__colsample_bytree': np.float64(0.7988994023569542), 'clf__learning_rate': np.float64(0.06716687886518624), 'clf__max_depth': 2, 'clf__min_child_weight': 8, 'clf__n_estimators': 180, 'clf__subsample': np.float64(0.706712405710114)} 0.865


# MLP

In [50]:
rows = []
for fold, (fit_idx, fold_val_idx) in enumerate(cv.split(X_train, y_train)):
    X_fit, y_fit = X_train.iloc[fit_idx], y_train.iloc[fit_idx]
    X_fold_val, y_fold_val = X_train.iloc[fold_val_idx], y_train.iloc[fold_val_idx]

    prep = build_preprocessor().fit(X_fit)
    X_inner, X_stop, y_inner, y_stop = train_test_split(
        X_fit, y_fit, test_size=0.2, stratify=y_fit, random_state=SEED
    )
    model, best_epoch = train_mlp(
        prep.transform(X_inner), y_inner, prep.transform(X_stop), y_stop, seed=SEED
    )

    proba = predict_proba(model, prep.transform(X_fold_val))
    pred = (proba >= 0.5).astype(int)
    rows.append({
        "roc_auc": roc_auc_score(y_fold_val, proba),
        "recall": recall_score(y_fold_val, pred),
        "precision": precision_score(y_fold_val, pred),
        "best_epoch": best_epoch,
    })

pd.DataFrame(rows).round(3)

,roc_auc,recall,precision,best_epoch
0,0.855,0.542,0.675,31
1,0.840,0.538,0.603,12
2,0.853,0.575,0.672,25
3,0.871,0.602,0.698,43
4,0.865,0.505,0.702,24
